In [1]:
# -*- coding: utf-8 -*-
"""
Read dataset_stats.h5 and plot:
  For each dataset, for frames 0 and 19:
    1) Global histogram (all cells x all records) with mean/std/range/min/max annotated in title
    2) Per-cell heatmaps: mean, std, min, max
Each dataset gets its own output folder.
"""

from pathlib import Path
import h5py
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm

# ======= user params =======
H5_PATH   = Path("./dataset_stats.h5")       # 你的统计 HDF5
OUT_DIR   = Path("./dataset_stats_plots")    # 输出图片根目录
FRAMES    = (0, 19)                          # 要绘制的帧
DPI       = 150
# ===========================

def _fmt(v: float) -> str:
    return f"{v:.4g}"

def _ensure_dir(p: Path):
    p.mkdir(parents=True, exist_ok=True)

def plot_global_hist(out_dir: Path, frame: int, g_global):
    """使用保存的 hist counts 与 (vmin,vmax,nbins) attrs 画直方图"""
    # 读统计量
    mean  = float(g_global["mean"][()])
    std   = float(g_global["std"][()])
    vmin  = float(g_global["min"][()])
    vmax  = float(g_global["max"][()])
    grng  = float(g_global["range"][()])

    # 直方图信息
    g_hist = g_global["hist"]
    h_vmin = float(g_hist.attrs["vmin"])
    h_vmax = float(g_hist.attrs["vmax"])
    nbins  = int(g_hist.attrs["nbins"])
    counts = np.array(g_hist["counts"], dtype=np.int64)
    edges  = np.linspace(h_vmin, h_vmax, nbins + 1)
    centers = 0.5 * (edges[:-1] + edges[1:])
    widths  = (edges[1:] - edges[:-1])

    fig = plt.figure(figsize=(8,5))
    ax = plt.gca()
    ax.bar(centers, counts, width=widths, align="center", edgecolor="none")
    ax.set_xlabel("Value")
    ax.set_ylabel("Count")
    title = (f"Frame {frame} | Global Histogram\n"
             f"mean={_fmt(mean)}  std={_fmt(std)}  range=[{_fmt(vmin)}, {_fmt(vmax)}] (Δ={_fmt(grng)})")
    ax.set_title(title)
    ax.grid(True, alpha=0.3, linestyle="--")
    fig.tight_layout()
    fig.savefig(out_dir / f"frame{frame}_global_hist.png", dpi=DPI, bbox_inches="tight")
    plt.close(fig)

def _imshow2d(ax, arr, title):
    im = ax.imshow(arr, origin="lower", interpolation="nearest")
    ax.set_title(title, pad=6)
    ax.set_xticks([]); ax.set_yticks([])
    cbar = plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    cbar.ax.tick_params(labelsize=8)

def plot_per_cell_maps(out_dir: Path, frame: int, g_cell):
    """绘制 per-cell 的 mean/std/min/max 4 张热图（同一画布 2x2）"""
    mean = np.array(g_cell["mean"])
    std  = np.array(g_cell["std"])
    vmin = np.array(g_cell["min"])
    vmax = np.array(g_cell["max"])

    fig, axes = plt.subplots(2, 2, figsize=(12, 10))
    _imshow2d(axes[0,0], mean, "Per-cell mean")
    _imshow2d(axes[0,1], std,  "Per-cell std")
    _imshow2d(axes[1,0], vmin, "Per-cell min")
    _imshow2d(axes[1,1], vmax, "Per-cell max")
    fig.suptitle(f"Frame {frame} | Per-cell statistics", y=0.995, fontsize=12)
    fig.tight_layout(rect=[0,0,1,0.98])
    fig.savefig(out_dir / f"frame{frame}_percell_maps.png", dpi=DPI, bbox_inches="tight")
    plt.close(fig)

def main():
    if not H5_PATH.exists():
        raise FileNotFoundError(f"HDF5 not found: {H5_PATH}")

    _ensure_dir(OUT_DIR)
    with h5py.File(H5_PATH, "r") as h5:
        files_grp = h5["files"]
        datasets = list(files_grp.keys())
        print(f"[INFO] Found {len(datasets)} datasets in HDF5.")
        for i, name in enumerate(datasets, 1):
            fgrp = files_grp[name]
            # 每个数据集一个输出目录
            ds_out = OUT_DIR / name
            _ensure_dir(ds_out)

            # 友好提示
            N = int(fgrp.attrs.get("N", -1))
            H = int(fgrp.attrs.get("H", -1))
            W = int(fgrp.attrs.get("W", -1))
            print(f"[{i}/{len(datasets)}] {name}  shape: N={N}, H={H}, W={W}")

            for frame in FRAMES:
                g_frame = fgrp["frames"].get(str(frame))
                if g_frame is None:
                    print(f"  [WARN] frame {frame} not found for dataset {name}, skip.")
                    continue

                # 1) 全局直方图
                plot_global_hist(ds_out, frame, g_frame["global"])

                # 2) 逐 cell 四图
                plot_per_cell_maps(ds_out, frame, g_frame["per_cell"])

    print(f"\n[OK] Plots saved under: {OUT_DIR.resolve()}")

if __name__ == "__main__":
    main()


[INFO] Found 15 datasets in HDF5.
[1/15] dim2d_nx256_N1150_solver=exponax_nu0.000_t20.0_train_all_frames_modespec=wwwwwwwwww_norm2_alpha10_epsilon13.1072_steps10  shape: N=1150, H=256, W=256
[2/15] dim2d_nx256_N1150_solver=exponax_nu0.000_t20.0_train_all_frames_modespec=wwwwwwwwww_norm2_alpha10_epsilon39.3216_steps10  shape: N=1150, H=256, W=256
[3/15] dim2d_nx256_N1150_solver=exponax_nu0.000_t20.0_train_all_frames_modespec=wwwwwwwwww_norm2_alpha10_epsilon78.6432_steps10  shape: N=1150, H=256, W=256
[4/15] dim2d_nx256_N1150_solver=exponax_nu0.000_t20.0_train_all_frames_modespec=wwwwwwwwww_norm2_alpha1_epsilon13.1072_steps10  shape: N=1150, H=256, W=256
[5/15] dim2d_nx256_N1150_solver=exponax_nu0.000_t20.0_train_all_frames_modespec=wwwwwwwwww_norm2_alpha1_epsilon39.3216_steps10  shape: N=1150, H=256, W=256
[6/15] dim2d_nx256_N1150_solver=exponax_nu0.000_t20.0_train_all_frames_modespec=wwwwwwwwww_norm2_alpha1_epsilon78.6432_steps10  shape: N=1150, H=256, W=256
[7/15] dim2d_nx256_N1150_so

In [1]:
import pandas as pd

df = pd.read_csv("similarity_spatial_frame0.csv")
df

,target_name,target_type,target_path,target_dtype,target_solver,target_nx,target_N,target_nu,target_t,target_ntimepoints,...,hist_range_overlap_ratio,hist_kl_pq_nat,hist_kl_qp_nat,global_mean_absdiff,global_std_absdiff,global_q50_absdiff,mean_map_pearson_r,mean_map_norm_mae,std_map_pearson_r,std_map_norm_mae
0,dim2d_nx256_N1150_solver=exponax_nu0.000_t20.0...,train,/blue/shiboli.fsu/yifeisun.umich/adversarial_r...,NaN,exponax,256,1150,0.0,20.0,NaN,...,0.081692,22.960820,19.253213,1.260030,0.210379,1.259359,0.142269,0.850178,-0.036229,0.748697
1,dim2d_nx256_N1150_solver=exponax_nu0.000_t20.0...,train,/blue/shiboli.fsu/yifeisun.umich/adversarial_r...,NaN,exponax,256,1150,0.0,20.0,NaN,...,0.317965,3.679379,0.503338,0.006923,0.168765,0.007623,0.156708,0.286191,-0.012299,0.622779
2,dim2d_nx256_N1150_solver=exponax_nu0.000_t20.0...,train,/blue/shiboli.fsu/yifeisun.umich/adversarial_r...,NaN,exponax,256,1150,0.0,20.0,NaN,...,0.397456,12.201402,1.895123,0.490153,0.147675,0.489668,0.028325,0.614822,-0.143499,0.557436
3,dim2d_nx256_N1150_solver=exponax_nu0.000_t20.0...,train,/blue/shiboli.fsu/yifeisun.umich/adversarial_r...,NaN,exponax,256,1150,0.0,20.0,NaN,...,0.105664,22.940514,14.504842,1.256674,0.208480,1.259652,0.338534,0.846057,-0.042718,0.748155
4,dim2d_nx256_N1150_solver=exponax_nu0.000_t20.0...,train,/blue/shiboli.fsu/yifeisun.umich/adversarial_r...,NaN,exponax,256,1150,0.0,20.0,NaN,...,0.794912,0.006188,0.002448,0.013955,0.003427,0.016514,-0.160618,0.192603,-0.077372,0.119537
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
24475,dim2d_nx256_N50_solver=exponax_nu0.000_t20.0_t...,test,/blue/shiboli.fsu/yifeisun.umich/adversarial_r...,NaN,exponax,256,50,0.0,20.0,NaN,...,0.387233,3.473888,0.440942,0.139650,0.094741,0.191575,0.097201,0.336732,0.030024,0.343566
24476,dim2d_nx256_N50_solver=exponax_nu0.000_t20.0_t...,test,/blue/shiboli.fsu/yifeisun.umich/adversarial_r...,NaN,exponax,256,50,0.0,20.0,NaN,...,0.440495,10.856101,1.041797,0.314008,0.057291,0.265772,-0.292565,0.511887,-0.121976,0.248688
24477,dim2d_nx256_N50_solver=exponax_nu0.000_t20.0_t...,test,/blue/shiboli.fsu/yifeisun.umich/adversarial_r...,NaN,exponax,256,50,0.0,20.0,NaN,...,0.052674,22.804929,22.639776,1.336651,0.164626,1.365602,0.144633,0.814130,0.171794,0.559449
24478,dim2d_nx256_N50_solver=exponax_nu0.000_t20.0_t...,test,/blue/shiboli.fsu/yifeisun.umich/adversarial_r...,NaN,exponax,256,50,0.0,20.0,NaN,...,0.910332,0.612201,1.252077,0.320165,0.190949,0.419216,-0.104956,0.406031,-0.246768,0.366946


In [2]:
df.columns

Index(['target_name', 'target_type', 'target_path', 'target_dtype',
       'target_solver', 'target_nx', 'target_N', 'target_nu', 'target_t',
       'target_ntimepoints', 'target_length_scale', 'target_variance',
       'target_alpha', 'target_tau', 'target_period', 'target_exp_factor',
       'target_vmax', 'target_vmin', 'gen_name', 'gen_type', 'gen_path',
       'gen_dtype', 'gen_solver', 'gen_nx', 'gen_N', 'gen_nu', 'gen_t',
       'gen_ntimepoints', 'gen_length_scale', 'gen_variance', 'gen_alpha',
       'gen_tau', 'gen_period', 'gen_exp_factor', 'gen_vmax', 'gen_vmin', 'H',
       'W', 'grid_bins', 'hist_js_bits', 'hist_tv', 'hist_emd',
       'hist_emd_norm', 'hist_overlap_coeff', 'hist_range_overlap_ratio',
       'hist_kl_pq_nat', 'hist_kl_qp_nat', 'global_mean_absdiff',
       'global_std_absdiff', 'global_q50_absdiff', 'mean_map_pearson_r',
       'mean_map_norm_mae', 'std_map_pearson_r', 'std_map_norm_mae'],
      dtype='object')

In [15]:
# -*- coding: utf-8 -*-
from pathlib import Path
import re
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from matplotlib.cm import get_cmap

# ---- 基础配色（保持与之前一致） ----
GROUP_BASE_COLOR = {"test":"#ff7f0e","train":"#2ca02c","expanded":"#9467bd","other":"#7f7f7f"}
KERNEL_ORDER = ["gaussian", "rbf", "rq", "matern", "periodic", "other"]
KERNEL_HATCH = {"gaussian": "-", "rbf": ".", "rq": "\\", "matern": "/", "periodic": "x", "other": None}

def _sanitize_filename(s: str) -> str:
    s = re.sub(r"[^\w\-.]+", "_", str(s))
    return s[:230]

def _infer_target_type(row_type: str | None, target_name: str) -> str:
    t = (row_type or "").lower()
    n = (target_name or "").lower()
    if t in ("train","test","expanded"):
        return t
    if "expanded" in n: return "expanded"
    if re.search(r"(?:^|_)train(?:_|$)", n): return "train"
    if re.search(r"(?:^|_)test(?:_|$)",  n): return "test"
    return "other"

# —— 可选：从名称里“猜一猜” kernel，用来决定 hatch（猜不到就不加）——
def _parse_kernel_from_text(txt: str) -> str:
    txt = (txt or "").lower()
    for kw, canon in [
        ("gaussian","gaussian"), ("rbf","rbf"),
        ("rq","rq"), ("rational_quadratic","rq"),
        ("matern","matern"), ("periodic","periodic"),
    ]:
        if kw in txt: return canon
    return "other"

def _build_gen_palette(gen_names: list[str], cmap_name="coolwarm"):
    cmap = get_cmap(cmap_name)
    n = max(1, len(gen_names))
    vals = [0.5] if n == 1 else np.linspace(0.15, 0.85, n)  # 与之前的范围一致
    return {g: cmap(v) for g, v in zip(gen_names, vals)}

# —— 额外小工具（若文件里还没有的话就加上）——
def _canonical_range_tuple(lo, hi):
    def _f(x):
        try: return float(x)
        except: return np.nan
    lo = _f(lo); hi = _f(hi)
    if np.isnan(lo) or np.isnan(hi):
        return None
    return (lo, hi) if lo <= hi else (hi, lo)

def _range_label(rt):
    if rt is None: return "range=unknown"
    lo, hi = rt
    return f"range=({lo:g},{hi:g})"

def _build_range_palette(range_labels, cmap_name="coolwarm"):
    from matplotlib.cm import get_cmap
    labels = sorted(set(range_labels))
    if not labels:
        return {}
    cmap = get_cmap(cmap_name)
    lo, hi = 0.15, 0.85
    n = len(labels)
    vals = [0.5] if n == 1 else np.linspace(lo, hi, n)
    return {lab: cmap(v) for lab, v in zip(labels, vals)}

# ========= 新函数：与我上面给你的版本一致 =========
def _family_from_dtype_or_name(dtype: str | None, name: str | None) -> str | None:
    s = ((dtype or "") + " " + (name or "")).lower()
    if s.startswith("negloggrf") or " negloggrf" in s: return "NegLogGRF"
    if s.startswith("loggrf")    or " loggrf" in s:    return "LogGRF"
    if s.startswith("grf")       or " grf" in s:       return "GRF"
    return None

def _range_tuple(lo, hi):
    try:
        lo = float(lo); hi = float(hi)
        if not np.isfinite(lo) or not np.isfinite(hi): return None
        return (lo, hi) if lo <= hi else (hi, lo)
    except Exception:
        return None

def _range_label(rt):
    if rt is None: return "range=unknown"
    lo, hi = rt
    return f"range=({lo:g},{hi:g})"

def _build_range_palette(range_labels, cmap_name="coolwarm"):
    cm = get_cmap(cmap_name)
    uniq = sorted(set(range_labels))
    vals = [0.5] if len(uniq) == 1 else np.linspace(0.15, 0.85, len(uniq))
    return {rl: cm(v) for rl, v in zip(uniq, vals)}

def plot_similarity_bars(
    csv_path: str | Path,
    target_name: str,                   # 可以是具体名字，也可以是 'train' / 'test' / 'expanded'
    metric: str,
    save_root: str | Path = "plots/similarity",
    family: str | None = None,          # 'GRF' / 'LogGRF' / 'NegLogGRF'，None 表示不过滤
    fuzzy: bool = False,                # True: target_name 用包含式匹配
    aggregate_bar: str | None = "mean", # {'best','mean','median',None} 右侧汇总柱
    add_family_suffix_to_filename: bool = False,  # True → 文件名后缀加 __GRF 等以免覆盖
    dpi: int = 160,
):
    """
    与 ΔRMSE 图保持同一套“排序、尺寸、色彩、分段”的视觉：
      - 横轴顺序：kernel 顺序 -> range(按 lo,hi 升序) -> 原始顺序（稳定排序）
      - generalizability: coolwarm(按 range) + kernel hatch
      - 上下 x 轴刻度：上=每个 (kernel,range) 段；下=generalizability / target
      - 竖直虚线：段分界线 + gen/target 分界线
      - 图尺寸：width=max(20,int(Nbars*0.06)), height=13
      - 保存路径：与你之前完全一致
    """
    # --- 读表 & 选 target ---
    csv_path = Path(csv_path)
    if not csv_path.exists():
        raise FileNotFoundError(csv_path)
    df = pd.read_csv(csv_path, low_memory=False)

    need_cols = {"target_name","target_type","gen_name","gen_type",metric}
    miss = [c for c in need_cols if c not in df.columns]
    if miss:
        raise ValueError(f"CSV 缺少必要列: {miss}")

    tgt = str(target_name).strip()
    if tgt.lower() in {"train","test","expanded"}:
        sel_t = df["target_type"].astype(str).str.lower() == tgt.lower()
        tgt_group = tgt.lower()
        tgt_label_for_title = tgt
    else:
        tmask = df["target_name"].astype(str)
        sel_t = tmask.str.contains(tgt, regex=False, na=False) if fuzzy else (tmask == tgt)
        if not sel_t.any():
            raise ValueError(f"找不到 target_name=={tgt}（fuzzy={fuzzy}）。")
        tgt_group = (df.loc[sel_t, "target_type"].astype(str).str.lower().iloc[0]
                     if "target_type" in df.columns else "other")
        tgt_label_for_title = tgt

    sub = df.loc[sel_t].copy()
    sub = sub[sub["gen_type"].astype(str).str.lower() == "generalizability"].copy()
    if sub.empty:
        raise ValueError("筛完 gen_type=='generalizability' 后为空。")

    # --- family 过滤（GRF / LogGRF / NegLogGRF） ---
    def _family_from_dtype_or_name(dtype: str | None, name: str | None) -> str | None:
        s = ((dtype or "") + " " + (name or "")).lower()
        if s.startswith("negloggrf") or " negloggrf" in s: return "NegLogGRF"
        if s.startswith("loggrf")    or " loggrf" in s:    return "LogGRF"
        if s.startswith("grf")       or " grf" in s:       return "GRF"
        return None
    fam = None if family is None else str(family).strip().lower()
    if fam is not None:
        sub = sub[sub.apply(lambda r: ((_family_from_dtype_or_name(
            str(r.get("gen_dtype","")), str(r.get("gen_name",""))) or "").lower()) == fam, axis=1)]
        if sub.empty:
            raise ValueError(f"family={family} 下没有可用的 generalizability 行。")

    # --- 聚合到每个 gen，并补 kernel / range 信息 ---
    extra_cols = [c for c in ["gen_dtype","gen_vmin","gen_vmax"] if c in sub.columns]
    g = (sub.groupby(["gen_name"] + extra_cols, dropna=False)[metric]
             .mean().reset_index())

    # kernel from dtype or name（兜底）
    def _parse_kernel_from_text(txt: str) -> str:
        t = (txt or "").lower()
        for kw, canon in [("gaussian","gaussian"),("rbf","rbf"),("rq","rq"),
                          ("rational_quadratic","rq"),("matern","matern"),("periodic","periodic")]:
            if kw in t: return canon
        return "other"
    g["kernel"] = g.get("gen_dtype","").map(_parse_kernel_from_text)
    g["kernel"] = np.where(g["kernel"].isna() | (g["kernel"]=="other"),
                           g["gen_name"].map(_parse_kernel_from_text), g["kernel"])

    # range(vmin,vmax)
    def _rt(lo, hi):
        try:
            lo = float(lo); hi = float(hi)
            if not np.isfinite(lo) or not np.isfinite(hi): return None
            return (lo, hi) if lo <= hi else (hi, lo)
        except Exception:
            return None
    if "gen_vmin" in g.columns and "gen_vmax" in g.columns:
        g["range_tuple"] = [_rt(lo, hi) for lo, hi in zip(g["gen_vmin"], g["gen_vmax"])]
    else:
        pat1 = re.compile(r"vmin([\-0-9\.]+).*?vmax([\-0-9\.]+)", re.I)
        pat2 = re.compile(r"vmax([\-0-9\.]+).*?vmin([\-0-9\.]+)", re.I)
        tmp = []
        for nm in g["gen_name"].astype(str):
            m = pat1.search(nm) or pat2.search(nm)
            tmp.append(_rt(m.group(1), m.group(2)) if m else None)
        g["range_tuple"] = tmp
    def _rl(rt):
        if rt is None: return "range=unknown"
        lo, hi = rt;  return f"range=({lo:g},{hi:g})"
    g["range_label"] = [ _rl(rt) for rt in g["range_tuple"] ]

    # --- 固定次序（kernel->range->原始序；稳定排序） ---
    order_kernel = {k:i for i,k in enumerate(KERNEL_ORDER)}
    def _range_key(rt): return (np.inf, np.inf) if rt is None else (rt[0], rt[1])
    g["_ord_kernel"] = g["kernel"].map(lambda k: order_kernel.get(k, len(KERNEL_ORDER)))
    g["_ord_range"]  = g["range_tuple"].map(_range_key)
    g["_ord_idx"]    = np.arange(len(g))
    g = g.sort_values(by=["_ord_kernel","_ord_range","_ord_idx"], ascending=True, kind="mergesort").reset_index(drop=True)

    # --- 段信息（用于 top-ticks 与虚线） ---
    segments = []   # [{start,end,kernel,range_label}]
    if len(g) > 0:
        cur = (g.loc[0,"kernel"], g.loc[0,"range_label"]); seg_start = 0
        for i in range(1, len(g)):
            key = (g.loc[i,"kernel"], g.loc[i,"range_label"])
            if key != cur:
                segments.append({"start": seg_start, "end": i-1,
                                 "kernel": cur[0], "range_label": cur[1]})
                cur = key; seg_start = i
        segments.append({"start": seg_start, "end": len(g)-1,
                         "kernel": cur[0], "range_label": cur[1]})
    # top x-axis ticks
    top_ticks  = [ (seg["start"] + seg["end"]) / 2.0 for seg in segments ]
    top_labels = []
    for seg in segments:
        rg = seg["range_label"] or "range=unknown"
        rg_short = re.sub(r"^range=","",rg)
        top_labels.append(f"{seg['kernel']}\n{rg_short}")
    # 段内分界线（在两个段之间的 0.5 处）
    seg_boundaries = [ seg["end"] + 0.5 for seg in segments[:-1] ]

    # --- 颜色 & hatch 与 ΔRMSE 一致 ---
    from matplotlib.cm import get_cmap
    def _build_range_palette(range_labels, cmap_name="coolwarm"):
        cm = get_cmap(cmap_name)
        uniq = sorted(set(range_labels))
        vals = [0.5] if len(uniq)==1 else np.linspace(0.15,0.85,len(uniq))
        return {rl: cm(v) for rl, v in zip(uniq, vals)}
    pal_range = _build_range_palette(g["range_label"].tolist(), cmap_name="coolwarm")
    colors  = [pal_range[rl] for rl in g["range_label"]]
    hatches = [KERNEL_HATCH.get(k, None) for k in g["kernel"]]

    # --- 目标聚合柱 ---
    vals_target = pd.to_numeric(sub[metric], errors="coerce").replace([np.inf,-np.inf], np.nan).dropna()
    agg_val = None
    if aggregate_bar in {"best","mean","median"} and len(vals_target) > 0:
        name = metric.lower()
        low_better  = any(k in name for k in ["div","tv","emd","absdiff","js","kl","mae"])
        high_better = any(k in name for k in ["overlap","pearson","corr","r2"])
        if aggregate_bar == "best":
            agg_val = float(vals_target.min() if (low_better and not high_better) else vals_target.max())
        elif aggregate_bar == "mean":
            agg_val = float(vals_target.mean())
        elif aggregate_bar == "median":
            agg_val = float(vals_target.median())

    x_names = g["gen_name"].tolist()
    y_vals  = g[metric].astype(float).tolist()
    if agg_val is not None and np.isfinite(agg_val):
        x_names.append(f"{tgt_group}({aggregate_bar})")
        y_vals.append(agg_val)
        colors.append(GROUP_BASE_COLOR.get(tgt_group, GROUP_BASE_COLOR["other"]))
        hatches.append(None)
        gen_target_boundary = len(g) - 0.5
    else:
        gen_target_boundary = None

    # --- 尺寸 & 绘图（同 ΔRMSE） ---
    n_bars = len(y_vals)
    fig_w = max(20, int(n_bars * 0.06))
    fig, ax = plt.subplots(figsize=(fig_w, 13))

    xs = np.arange(n_bars)
    bars = ax.bar(xs, y_vals, color=colors, edgecolor="black", linewidth=0.0, align="center")
    for b, h in zip(bars, hatches):
        if h: b.set_hatch(h)

    ax.set_xlim(-0.5, n_bars - 0.5)
    ax.margins(x=0)
    ax.set_axisbelow(True)
    ax.grid(axis='y', linestyle='--', linewidth=1.0, alpha=0.6)
    ax.axhline(0.0, color="black", lw=1.0, alpha=0.8)
    ax.set_ylabel(metric, fontsize=12)

    # --- 上下 x-ticks & 分界虚线 ---
    # bottom：两块（generalizability / target）
    bot_ticks = []
    bot_labels = []
    if len(g) > 0:
        bot_ticks.append((0 + (len(g)-1)) / 2.0)
        bot_labels.append("generalizability")
    if gen_target_boundary is not None:
        bot_ticks.append(len(y_vals)-1)
        bot_labels.append(tgt_group)
    ax.set_xticks(bot_ticks)
    ax.set_xticklabels(bot_labels, fontsize=11)

    # top：每个 (kernel,range) 段
    ax_top = ax.secondary_xaxis('top')
    ax_top.set_xticks(top_ticks)
    ax_top.set_xticklabels(top_labels, fontsize=10)
    ax_top.tick_params(axis='x', labelrotation=90, length=0, pad=6)

    # 段分界虚线
    for xb in seg_boundaries:
        ax.axvline(xb, color="black", lw=0.8, ls="--", alpha=0.6)
    # gen / target 分界虚线（粗一点）
    if gen_target_boundary is not None:
        ax.axvline(gen_target_boundary, color="black", lw=1.2, ls="--", alpha=0.8)

    # 标题
    title_family = f" | family={family}" if family else ""
    ax.set_title(f"{tgt_label_for_title} — {metric} vs. generalizability{title_family}", fontsize=14, pad=24)
    plt.subplots_adjust(top=0.78)

    # 图例（range 色块 + kernel hatch + 目标类型）
    legend_elems = []
    for rl in sorted(set(g["range_label"])):
        legend_elems.append(Patch(facecolor=pal_range[rl], edgecolor="none", label=f"generalizability\n{rl}"))
    for k in [kk for kk in KERNEL_ORDER if kk in set(g["kernel"])]:
        hh = KERNEL_HATCH.get(k)
        if hh:
            legend_elems.append(Patch(facecolor="white", edgecolor="black", hatch=hh, linewidth=0.0, label=f"kernel={k}"))
    if agg_val is not None:
        legend_elems.append(Patch(facecolor=GROUP_BASE_COLOR.get(tgt_group, GROUP_BASE_COLOR["other"]),
                                  edgecolor="none", label=tgt_group))
    if legend_elems:
        ax.legend(handles=legend_elems, loc="upper left", bbox_to_anchor=(1.01, 1.0), title="Legend")

    # ===== 保存（严格按你原来的 5 行）=====
    save_dir = Path(save_root) / _sanitize_filename(target_name)
    save_dir.mkdir(parents=True, exist_ok=True)
    fname = _sanitize_filename(metric) + (f"__{family}" if (family and add_family_suffix_to_filename) else "")
    out_png = save_dir / f"{fname}.png"
    fig.savefig(out_png, dpi=dpi, bbox_inches="tight")
    plt.close(fig)
    print(f"[OK] saved: {out_png}")

    # 返回按优劣排序的前 10 个 gen（不影响横轴绘图顺序）
    name = metric.lower()
    low_better  = any(k in name for k in ["div","tv","emd","absdiff","js","kl","mae"])
    high_better = any(k in name for k in ["overlap","pearson","corr","r2"])
    asc = True if low_better and not high_better else False
    nearest = g.set_index("gen_name")[metric].astype(float).sort_values(ascending=asc).head(10).reset_index()
    nearest = nearest.rename(columns={"gen_name":"nearest_gen", metric:"metric"})
    return nearest

def plot_similarity_bars_by_family(
    csv_path: str | Path,
    target_name: str,
    metric: str,
    save_root: str | Path = "plots/similarity",
    **kwargs
):
    """一次性画三套（GRF/LogGRF/NegLogGRF），文件名自动加后缀避免覆盖。"""
    for fam in ["GRF","LogGRF","NegLogGRF"]:
        plot_similarity_bars(
            csv_path=csv_path,
            target_name=target_name,
            metric=metric,
            save_root=save_root,
            family=fam,
            add_family_suffix_to_filename=True,  # 避免覆盖
            **kwargs
        )



In [18]:
df["target_name"].unique()

array(['dim2d_nx256_N1150_solver=exponax_nu0.000_t20.0_train_all_frames',
       'dim2d_nx256_N1150_solver=exponax_nu0.000_t20.0_train_all_frames_modespec=wwwwwwwwww_norm2_alpha10_epsilon13.1072_steps10',
       'dim2d_nx256_N1150_solver=exponax_nu0.000_t20.0_train_all_frames_modespec=wwwwwwwwww_norm2_alpha10_epsilon39.3216_steps10',
       'dim2d_nx256_N1150_solver=exponax_nu0.000_t20.0_train_all_frames_modespec=wwwwwwwwww_norm2_alpha10_epsilon78.6432_steps10',
       'dim2d_nx256_N1150_solver=exponax_nu0.000_t20.0_train_all_frames_modespec=wwwwwwwwww_norm2_alpha1_epsilon13.1072_steps10',
       'dim2d_nx256_N1150_solver=exponax_nu0.000_t20.0_train_all_frames_modespec=wwwwwwwwww_norm2_alpha1_epsilon39.3216_steps10',
       'dim2d_nx256_N1150_solver=exponax_nu0.000_t20.0_train_all_frames_modespec=wwwwwwwwww_norm2_alpha1_epsilon78.6432_steps10',
       'dim2d_nx256_N1150_solver=exponax_nu0.000_t20.0_train_all_frames_modespec=wwwwwwwwww_norm2_alpha2.5_epsilon13.1072_steps10',
       'dim

In [17]:
df.columns

Index(['target_name', 'target_type', 'target_path', 'target_dtype',
       'target_solver', 'target_nx', 'target_N', 'target_nu', 'target_t',
       'target_ntimepoints', 'target_length_scale', 'target_variance',
       'target_alpha', 'target_tau', 'target_period', 'target_exp_factor',
       'target_vmax', 'target_vmin', 'gen_name', 'gen_type', 'gen_path',
       'gen_dtype', 'gen_solver', 'gen_nx', 'gen_N', 'gen_nu', 'gen_t',
       'gen_ntimepoints', 'gen_length_scale', 'gen_variance', 'gen_alpha',
       'gen_tau', 'gen_period', 'gen_exp_factor', 'gen_vmax', 'gen_vmin', 'H',
       'W', 'grid_bins', 'hist_js_bits', 'hist_tv', 'hist_emd',
       'hist_emd_norm', 'hist_overlap_coeff', 'hist_range_overlap_ratio',
       'hist_kl_pq_nat', 'hist_kl_qp_nat', 'global_mean_absdiff',
       'global_std_absdiff', 'global_q50_absdiff', 'mean_map_pearson_r',
       'mean_map_norm_mae', 'std_map_pearson_r', 'std_map_norm_mae'],
      dtype='object')

In [16]:
plot_similarity_bars_by_family(
    csv_path="similarity_spatial_frame0.csv",
    target_name="dim2d_nx256_N1150_solver=exponax_nu0.000_t20.0_train_all_frames_modespec=wwwwwwwwww_norm2_alpha10_epsilon13.1072_steps10",
    metric='hist_js_bits',
    save_root = "plots/similarity",
)

/tmp/ipykernel_3624590/1113927299.py:237: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  cm = get_cmap(cmap_name)


[OK] saved: plots/similarity/dim2d_nx256_N1150_solver_exponax_nu0.000_t20.0_train_all_frames_modespec_wwwwwwwwww_norm2_alpha10_epsilon13.1072_steps10/hist_js_bits__GRF.png


/tmp/ipykernel_3624590/1113927299.py:237: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  cm = get_cmap(cmap_name)


[OK] saved: plots/similarity/dim2d_nx256_N1150_solver_exponax_nu0.000_t20.0_train_all_frames_modespec_wwwwwwwwww_norm2_alpha10_epsilon13.1072_steps10/hist_js_bits__LogGRF.png


/tmp/ipykernel_3624590/1113927299.py:237: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  cm = get_cmap(cmap_name)


[OK] saved: plots/similarity/dim2d_nx256_N1150_solver_exponax_nu0.000_t20.0_train_all_frames_modespec_wwwwwwwwww_norm2_alpha10_epsilon13.1072_steps10/hist_js_bits__NegLogGRF.png


In [19]:
# 读一次 CSV（和你已有的 df 一样）
csv_path = "similarity_spatial_frame0.csv"
df = pd.read_csv(csv_path, low_memory=False)

# 1) 要画哪些 metric（按你表里的列）
metrics = [
    "hist_js_bits","hist_tv","hist_emd","hist_emd_norm",
    "hist_overlap_coeff","hist_range_overlap_ratio",
    "hist_kl_pq_nat","hist_kl_qp_nat",
    "global_mean_absdiff","global_std_absdiff","global_q50_absdiff",
    "mean_map_pearson_r","mean_map_norm_mae",
    "std_map_pearson_r","std_map_norm_mae",
]

# 2) 遍历哪些 target（你给的 df['target_name'].unique()）
targets = pd.Series(df["target_name"]).dropna().unique()

# —— 双重循环：metric × target ——
for metric in metrics:
    if metric not in df.columns:
        print(f"[skip metric] {metric} 不在 CSV 列里")
        continue
    for t in targets:
        try:
            print(f"[run] metric={metric} | target={t}")
            plot_similarity_bars_by_family(
                csv_path=csv_path,
                target_name=t,
                metric=metric,
                save_root="plots/similarity",  # 内部依旧保存到  plots/similarity/<target>/<metric>.png
            )
        except Exception as e:
            print(f"[skip] metric={metric} | target={t} -> {e}")

[run] metric=hist_js_bits | target=dim2d_nx256_N1150_solver=exponax_nu0.000_t20.0_train_all_frames


/tmp/ipykernel_3624590/1113927299.py:237: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  cm = get_cmap(cmap_name)


[OK] saved: plots/similarity/dim2d_nx256_N1150_solver_exponax_nu0.000_t20.0_train_all_frames/hist_js_bits__GRF.png


/tmp/ipykernel_3624590/1113927299.py:237: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  cm = get_cmap(cmap_name)


[OK] saved: plots/similarity/dim2d_nx256_N1150_solver_exponax_nu0.000_t20.0_train_all_frames/hist_js_bits__LogGRF.png


...

[OK] saved: plots/similarity/dim2d_nx256_N1150_solver_exponax_nu0.000_t20.0_train_all_frames_modespec_wwwwwwwwww_norm2_alpha10_epsilon78.6432_steps10/global_mean_absdiff__NegLogGRF.png
[run] metric=global_mean_absdiff | target=dim2d_nx256_N1150_solver=exponax_nu0.000_t20.0_train_all_frames_modespec=wwwwwwwwww_norm2_alpha1_epsilon13.1072_steps10


/tmp/ipykernel_3624590/1113927299.py:237: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  cm = get_cmap(cmap_name)


[OK] saved: plots/similarity/dim2d_nx256_N1150_solver_exponax_nu0.000_t20.0_train_all_frames_modespec_wwwwwwwwww_norm2_alpha1_epsilon13.1072_steps10/global_mean_absdiff__GRF.png


/tmp/ipykernel_3624590/1113927299.py:237: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  cm = get_cmap(cmap_name)


KeyboardInterrupt: 

In [ ]:
# 单个 family
plot_similarity_bars("similarity_spatial_frame0.csv", "train", "hist_js_bits", family="GRF")
